# Setwise Toy Experiments

This notebook is a trimmed version of `rag_frameworks_progress.ipynb` for the toy setwise experiments only. It reuses the same Drive-mounted BM25/HotpotQA setup and imports `setwise_toy_utils.py` from the repo root.

Suggested rerun order in a fresh Colab runtime:
1. Mount Drive / `cd` into the repo
2. Run the BEIR HotpotQA + BM25 candidate preparation cells
3. Run the reranker helper cells
4. Run the toy setwise cells


## Runtime Guide

Use this as the recovery map if the Colab runtime expires. `GPU` means the cell benefits from or requires GPU; most cells here are CPU-safe.

- `from google.colab import drive`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `# one-time only` / `git clone https://github.com/Joseph2718/FlashRAG.git`: GPU `No`; rerun after expiry `Only if the repo is missing in Drive`; skip after expiry `Yes` if already cloned.
- `!ls /content/drive/MyDrive/FlashRAG/examples/quick_start/indexes/`: GPU `No`; rerun after expiry `No`; skip after expiry `Yes`.
- `%cd /content/drive/MyDrive/FlashRAG` / `!pip install -e .`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `!python scripts/setup_bm25_hotpotqa.py --download_dataset ...`: GPU `No`; rerun after expiry `Only if dataset/index are not already prepared`; skip after expiry `Yes` if setup already exists.
- `!pip install faiss-cpu`: GPU `No`; rerun after expiry `Usually no`; skip after expiry `Yes` unless a package reset happened.
- `# Optional GPU packages` (faiss-gpu, vllm): GPU `Yes` for vllm; rerun after expiry `No`; skip after expiry `Yes`.
- `# from huggingface_hub import login`: GPU `No`; rerun after expiry `No`; skip after expiry `Yes` unless downloading gated models.
- `# %cd /content/drive/MyDrive/FlashRAG/examples/methods` / `snapshot_download(...)`: GPU `No`; rerun after expiry `No`; skip after expiry `Yes` unless downloading the Llama checkpoint.
- `# !python run_exp.py --method_name bm25-naive ...`: GPU `No` for BM25 itself`; rerun after expiry `Optional`; skip after expiry `Yes` if using the BEIR notebook path instead.
- `# from huggingface_hub import hf_hub_download`: GPU `No`; rerun after expiry `No`; skip after expiry `Yes` unless rebuilding the wiki corpus from HF.
- `# one-time run` / `python -m flashrag.retriever.index_builder ... wiki18_100w.jsonl`: GPU `No`; rerun after expiry `Only if the large BM25 wiki index is missing`; skip after expiry `Yes` if index already exists.
- `!mkdir -p /content/FlashRAG_local/...` / `!cp ...`: GPU `No`; rerun after expiry `Yes` because `/content` is ephemeral`; skip after expiry `No` if later cells expect `/content/FlashRAG_local`.
- `## BEIR HotpotQA qrels with FlashRAG BM25`: markdown only; this starts the important reproducible path.
- `# If needed in a fresh Colab runtime:` / BEIR install/import cell: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `import json` / BEIR dataset loading: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `%cd /content/drive/MyDrive/FlashRAG`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `import pandas as pd`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `!sudo apt-get update -qq`: GPU `No`; rerun after expiry `Usually no`; skip after expiry `Yes` if `pyserini` is already working.
- `!pip -q install pyserini`: GPU `No`; rerun after expiry `Only if the package disappeared`; skip after expiry `Yes` if already installed.
- `from beir import util`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `## Listwise REINFORCE on Fixed BM25 Candidates`: markdown only; this starts the toy reranking experiment.
- `import numpy as np`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No`.
- `def _discounts(k, device, dtype=torch.float32):`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No` because later cells depend on these helpers.
- `class TinyListwiseRanker(nn.Module):`: GPU `No`; rerun after expiry `Yes`; skip after expiry `No` because later cells depend on these samplers/losses.
- `torch.manual_seed(RERANK_SEED)`: GPU `No`; rerun after expiry `Optional`; skip after expiry `Yes` if you do not need the smoke-test prints.
- `def flatten_grads(model):`: GPU `No`; rerun after expiry `Yes` if you want any variance plots`; skip after expiry `Yes` if you only want training.
- `whole_df = variance_df[variance_df["reward_mode"] == "whole"].copy()`: GPU `No`; rerun after expiry `Optional`; skip after expiry `Yes` if the full three-estimator plot is enough.
- `plt.figure(figsize=(8, 5))` / three-estimator variance comparison: GPU `No`; rerun after expiry `Optional but recommended`; skip after expiry `Yes` if `variance_df` is already saved or visible.
- `TRAIN_STEPS = 60`: GPU `No` for correctness, `Yes` only if you want it faster`; rerun after expiry `Optional`; skip after expiry `Yes` unless you want the toy learning curve.
- `<empty>` final cell: currently blank; no rerun needed.

### Minimal rerun path after a runtime reset

If you only want the BEIR BM25 + toy listwise experiment back, rerun this subset in order:

1. Drive mount.
2. Editable FlashRAG install cell.
3. `/content/FlashRAG_local` copy cell if you use those local paths.
4. The BEIR HotpotQA block from `# If needed in a fresh Colab runtime:` through `from beir import util`.
5. The listwise block from `import numpy as np` through the cells you care about: smoke test, variance plots, and/or training.

### Cells that can usually be skipped after expiry

- Repo clone, if the repo already exists in Drive.
- Large wiki BM25 index build, if `indexes_wiki/bm25` already exists.
- Hugging Face login/model download/vLLM cells.

### GPU note

Nothing in the BEIR BM25 block or the listwise reranking block strictly requires GPU. The toy reranker, variance sweeps, and training loop all run on CPU; GPU only makes the longer sweeps finish faster.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/FlashRAG

!pip install -e .
!pip install termcolor huggingface_hub

## BEIR HotpotQA qrels with FlashRAG BM25

This section uses **BEIR HotpotQA** for `queries` and `qrels`, while still using **FlashRAG** to build the BM25 index and run retrieval.

Why this works:
- BEIR already provides `qrels`, so we do not need to define relevance heuristically.
- FlashRAG can still be the retrieval framework, as long as we build the index over the **same BEIR corpus** and preserve the BEIR document ids.
- Evaluation is then done with BEIR's `EvaluateRetrieval`, using the retrieved document ids returned by FlashRAG.

Important:
- This is a separate path from the earlier `wiki18_100w` experiment.
- Do **not** mix BEIR `qrels` with the `wiki18_100w` corpus; the document ids will not match.
- For a quick smoke test, use a small `BEIR_N_QUERIES`; for final results, set it to `None`.

In [ ]:
# If needed in a fresh Colab runtime:
!pip install -q beir

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

import json
import os
import pathlib

BEIR_DATASET = "hotpotqa"
BEIR_SPLIT = "test"      # change to "dev" or "train" if you want
BEIR_N_QUERIES = None      # set to None for the full split

beir_root = "/content/drive/MyDrive/FlashRAG/beir_datasets"
os.makedirs(beir_root, exist_ok=True)

url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{BEIR_DATASET}.zip"
data_path = util.download_and_unzip(url, beir_root)

corpus, beir_queries_dict, beir_qrels = GenericDataLoader(data_folder=data_path).load(split=BEIR_SPLIT)

beir_qids = list(beir_queries_dict.keys())
if BEIR_N_QUERIES is not None:
    beir_qids = beir_qids[:BEIR_N_QUERIES]

beir_queries = [beir_queries_dict[qid] for qid in beir_qids]
beir_qrels_subset = {qid: beir_qrels[qid] for qid in beir_qids if qid in beir_qrels}

print(f"Loaded BEIR {BEIR_DATASET} split={BEIR_SPLIT}")
print(f"Corpus size: {len(corpus)}")
print(f"Queries selected: {len(beir_qids)}")

sample_qid = beir_qids[0]
print("\nSample qid:", sample_qid)
print("Sample query:", beir_queries_dict[sample_qid])
print("Sample qrels:", list(beir_qrels_subset[sample_qid].items())[:5])

In [ ]:
import json
import os
import subprocess
import sys

beir_flashrag_root = "/content/drive/MyDrive/FlashRAG/beir_hotpotqa_flashrag"
beir_corpus_path = os.path.join(beir_flashrag_root, f"{BEIR_DATASET}_{BEIR_SPLIT}_corpus.jsonl")
beir_index_root = os.path.join(beir_flashrag_root, f"indexes_{BEIR_DATASET}_{BEIR_SPLIT}")
beir_index_path = os.path.join(beir_index_root, "bm25")

os.makedirs(beir_flashrag_root, exist_ok=True)
os.makedirs(beir_index_root, exist_ok=True)

if not os.path.exists(beir_corpus_path):
    with open(beir_corpus_path, "w", encoding="utf-8") as f:
        for pid, doc in corpus.items():
            title = (doc.get("title") or "").strip()
            text = (doc.get("text") or "").strip()
            contents = f"{title}\n{text}" if title else text
            row = {
                "id": str(pid),
                "title": title,
                "text": text,
                "contents": contents,
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("Wrote FlashRAG-compatible BEIR corpus to:", beir_corpus_path)
else:
    print("FlashRAG-compatible BEIR corpus already exists:", beir_corpus_path)

if not os.path.exists(beir_index_path):
    cmd = [
        sys.executable,
        "-m",
        "flashrag.retriever.index_builder",
        "--retrieval_method",
        "bm25",
        "--corpus_path",
        beir_corpus_path,
        "--save_dir",
        beir_index_root,
        "--bm25_backend",
        "bm25s",
    ]
    print("Building FlashRAG BM25 index over the BEIR corpus...")
    subprocess.run(cmd, check=True)
else:
    print("BM25 index already exists:", beir_index_path)

In [ ]:
%cd /content/drive/MyDrive/FlashRAG

from flashrag.config import Config
from flashrag.utils import get_retriever

beir_config_dict = {
    "retrieval_method": "bm25",
    "bm25_backend": "bm25s",
    "index_path": beir_index_path,
    "corpus_path": beir_corpus_path,
    "retrieval_topk": 100,
    "save_note": f"beir-{BEIR_DATASET}-{BEIR_SPLIT}-bm25",
}

beir_config = Config(config_dict=beir_config_dict)
beir_retriever = get_retriever(beir_config)

flashrag_docs, flashrag_scores = beir_retriever.batch_search(beir_queries, return_score=True)

beir_results = {}
for qid, docs, scores in zip(beir_qids, flashrag_docs, flashrag_scores):
    doc_scores = {}
    for doc, score in zip(docs, scores):
        pid = str(doc["id"])
        doc_scores[pid] = max(doc_scores.get(pid, float("-inf")), float(score))
    beir_results[qid] = doc_scores

K_VALUES = [1, 3, 5, 10, 20]
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(beir_qrels_subset, beir_results, K_VALUES)

print("FlashRAG BM25 on BEIR HotpotQA")
print(json.dumps({
    "ndcg": ndcg,
    "map": _map,
    "recall": recall,
    "precision": precision,
}, indent=2))

sample_qid = beir_qids[0]
print("\nSample query:")
print(beir_queries_dict[sample_qid])
print("\nTop 5 retrieved docs:")
for rank, doc in enumerate(flashrag_docs[0][:5], start=1):
    print(rank, doc.get("id"), doc.get("title", ""))
print("\nGold qrels for sample query:")
print(list(beir_qrels_subset[sample_qid].items())[:10])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for metric_name, metric_dict in [
    ("nDCG", ndcg),
    ("MAP", _map),
    ("Recall", recall),
    ("Precision", precision),
]:
    for k_str, value in metric_dict.items():
        k = int(str(k_str).split("@")[1]) if "@" in str(k_str) else int(k_str)
        rows.append({"metric": metric_name, "k": k, "value": float(value)})

metrics_df = pd.DataFrame(rows).sort_values(["metric", "k"]).reset_index(drop=True)
display(metrics_df.pivot(index="k", columns="metric", values="value"))

plt.figure(figsize=(8, 5))
for metric_name in ["nDCG", "MAP", "Recall", "Precision"]:
    sub = metrics_df[metrics_df["metric"] == metric_name]
    plt.plot(sub["k"], sub["value"], marker="o", linewidth=2, label=metric_name)

plt.xticks(sorted(metrics_df["k"].unique()))
plt.ylim(0, 1)
plt.xlabel("k")
plt.ylabel("Score")
plt.title(f"FlashRAG BM25 on BEIR {BEIR_DATASET} ({BEIR_SPLIT}, N={len(beir_qids)})")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk
!java -version

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version

In [ ]:
!pip -q install pyserini

# Run BM25 over the BEIR HotpotQA test topics, using the prebuilt Lucene index.
!python -m pyserini.search.lucene \
  --threads 8 --batch-size 128 \
  --index beir-v1.0.0-hotpotqa.flat \
  --topics beir-v1.0.0-hotpotqa-test \
  --output run.hotpotqa.bm25-flat.txt \
  --output-format trec \
  --hits 1000 --bm25 --remove-query

In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

qrels2 = beir_qrels  # already loaded earlier

def load_trec_run(path):
    results = {}
    with open(path, "r") as f:
        for line in f:
            qid, _, docid, rank, score, _ = line.strip().split()
            results.setdefault(qid, {})[docid] = float(score)
    return results

lucene_results = load_trec_run("run.hotpotqa.bm25-flat.txt")

K_VALUES = [1,3,5,10,20,100]
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels2, lucene_results, K_VALUES)
print("Lucene BM25-flat NDCG@10:", ndcg["NDCG@10"])
print("Lucene BM25-flat NDCG:", {k: ndcg[f"NDCG@{k}"] for k in K_VALUES})

## Listwise REINFORCE on Fixed BM25 Candidates

This block keeps FlashRAG retrieval fixed and studies **listwise reranking variance** on top of the BEIR HotpotQA BM25 candidates already loaded above.

The progression is:
1. align the existing BM25 top-`M` candidates with BEIR qrels,
2. compute listwise `nDCG@k` rewards,
3. estimate policy gradients with vanilla REINFORCE,
4. swap in Gumbel perturb-and-sort sampling,
5. compare gradient variance against the number of Monte Carlo samples.
6. add an Oosterhuis-style `PL-Rank-1` estimator and test whether it reduces variance further.

This stays notebook-local on purpose so the sampling and reward logic is easy to inspect before extracting anything into library code.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

EXPERIMENT_TRACK = "richer_feature_estimator_benchmark"
VALID_EXPERIMENT_TRACKS = {
    "richer_feature_estimator_benchmark",
    "bm25_shortlist_demo",
}
assert EXPERIMENT_TRACK in VALID_EXPERIMENT_TRACKS

TRACK_TITLE_MAP = {
    "richer_feature_estimator_benchmark": "Richer-Feature Estimator Benchmark",
    "bm25_shortlist_demo": "BM25 Shortlist Demo",
}

RERANK_TOP_M = 20
RERANK_K = 10
RERANK_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RERANK_SEED = 7


def tokenize_text(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


def safe_doc_text(doc):
    title = str(doc.get("title", "") or "")
    contents = str(doc.get("contents", doc.get("text", "")) or "")
    return title, contents


def build_bm25_demo_features(scores, mask):
    safe_scores = torch.where(mask, scores, torch.zeros_like(scores))
    valid_scores = scores[mask]
    global_mean = valid_scores.mean()
    global_std = valid_scores.std().clamp_min(1e-6)
    normalized_bm25 = (safe_scores - global_mean) / global_std
    rank_positions = torch.arange(scores.shape[1], device=scores.device, dtype=scores.dtype).unsqueeze(0).expand_as(scores)
    reciprocal_rank = 1.0 / (rank_positions + 1.0)
    features = torch.stack([normalized_bm25, reciprocal_rank], dim=-1)
    features = features * mask.unsqueeze(-1)
    return features.to(dtype=torch.float64), ["normalized_bm25", "reciprocal_rank"]


def build_richer_rank_features(
    scores,
    mask,
    query_length,
    title_length,
    content_length,
    overlap_count,
    overlap_ratio,
    title_overlap,
):
    safe_scores = torch.where(mask, scores, torch.zeros_like(scores))
    mask_f = mask.to(dtype=scores.dtype)
    rank_positions = torch.arange(scores.shape[1], device=scores.device, dtype=scores.dtype).unsqueeze(0).expand_as(scores)
    reciprocal_rank = 1.0 / (rank_positions + 1.0)

    valid_scores = scores[mask]
    global_mean = valid_scores.mean()
    global_std = valid_scores.std().clamp_min(1e-6)
    normalized_bm25 = (safe_scores - global_mean) / global_std

    query_count = mask_f.sum(dim=1, keepdim=True).clamp_min(1.0)
    query_mean = (safe_scores * mask_f).sum(dim=1, keepdim=True) / query_count
    centered_scores = safe_scores - query_mean
    query_var = ((centered_scores.pow(2) * mask_f).sum(dim=1, keepdim=True) / query_count).clamp_min(1e-12)
    query_std = query_var.sqrt().clamp_min(1e-6)
    query_zscore = centered_scores / query_std

    query_min = scores.masked_fill(~mask, float("inf")).min(dim=1, keepdim=True).values
    query_max = scores.masked_fill(~mask, float("-inf")).max(dim=1, keepdim=True).values
    query_range = (query_max - query_min).clamp_min(1e-6)
    query_minmax = (safe_scores - query_min) / query_range

    top1_score = query_max
    gap_to_top1 = top1_score - safe_scores

    prev_scores = torch.cat([safe_scores[:, :1], safe_scores[:, :-1]], dim=1)
    next_scores = torch.cat([safe_scores[:, 1:], safe_scores[:, -1:]], dim=1)
    gap_to_prev = prev_scores - safe_scores
    gap_to_prev[:, 0] = 0.0
    gap_to_next = safe_scores - next_scores
    gap_to_next[:, -1] = 0.0

    top1_bucket = (rank_positions < 1).to(scores.dtype)
    top3_bucket = (rank_positions < 3).to(scores.dtype)
    top5_bucket = (rank_positions < 5).to(scores.dtype)
    top10_bucket = (rank_positions < 10).to(scores.dtype)

    candidate_list_length = query_count.expand_as(scores)
    availability = mask_f
    interaction = normalized_bm25 * reciprocal_rank

    features = torch.stack(
        [
            normalized_bm25,
            safe_scores,
            rank_positions,
            reciprocal_rank,
            gap_to_top1,
            gap_to_prev,
            gap_to_next,
            query_zscore,
            query_minmax,
            top1_bucket,
            top3_bucket,
            top5_bucket,
            top10_bucket,
            candidate_list_length,
            availability,
            interaction,
            query_length,
            title_length,
            content_length,
            overlap_count,
            overlap_ratio,
            title_overlap,
        ],
        dim=-1,
    )
    features = features * mask.unsqueeze(-1)
    feature_names = [
        "normalized_bm25",
        "raw_bm25",
        "rank_position",
        "reciprocal_rank",
        "gap_to_top1",
        "gap_to_prev",
        "gap_to_next",
        "query_zscore",
        "query_minmax",
        "top1_bucket",
        "top3_bucket",
        "top5_bucket",
        "top10_bucket",
        "candidate_list_length",
        "availability",
        "bm25_times_reciprocal_rank",
        "query_token_length",
        "title_token_length",
        "content_token_length",
        "lexical_overlap_count",
        "lexical_overlap_ratio",
        "title_overlap_flag",
    ]
    return features.to(dtype=torch.float64), feature_names


torch.manual_seed(RERANK_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RERANK_SEED)

candidate_doc_ids = []
candidate_scores_rows = []
candidate_labels_rows = []
candidate_mask_rows = []
candidate_query_length_rows = []
candidate_title_length_rows = []
candidate_content_length_rows = []
candidate_overlap_count_rows = []
candidate_overlap_ratio_rows = []
candidate_title_overlap_rows = []
active_qids = []

for qid, docs, scores in zip(beir_qids, flashrag_docs, flashrag_scores):
    docs = list(docs[:RERANK_TOP_M])
    scores = [float(s) for s in list(scores[:RERANK_TOP_M])]
    query_tokens = tokenize_text(beir_queries_dict[qid])
    query_token_set = set(query_tokens)
    query_length = float(len(query_tokens))

    doc_ids = [str(doc["id"]) for doc in docs]
    labels = [float(beir_qrels_subset.get(qid, {}).get(doc_id, 0.0)) for doc_id in doc_ids]
    title_lengths = []
    content_lengths = []
    overlap_counts = []
    overlap_ratios = []
    title_overlaps = []
    query_lengths = []

    for doc in docs:
        title, contents = safe_doc_text(doc)
        title_tokens = tokenize_text(title)
        content_tokens = tokenize_text(contents)
        doc_token_set = set(title_tokens) | set(content_tokens)
        overlap = float(len(query_token_set & doc_token_set))
        title_overlap = float(len(query_token_set & set(title_tokens)) > 0)

        title_lengths.append(float(len(title_tokens)))
        content_lengths.append(float(len(content_tokens)))
        overlap_counts.append(overlap)
        overlap_ratios.append(overlap / max(len(query_token_set), 1))
        title_overlaps.append(title_overlap)
        query_lengths.append(query_length)

    pad = RERANK_TOP_M - len(doc_ids)
    if pad > 0:
        doc_ids.extend([f"__pad__{qid}_{i}" for i in range(pad)])
        scores.extend([-1e9] * pad)
        labels.extend([0.0] * pad)
        title_lengths.extend([0.0] * pad)
        content_lengths.extend([0.0] * pad)
        overlap_counts.extend([0.0] * pad)
        overlap_ratios.extend([0.0] * pad)
        title_overlaps.extend([0.0] * pad)
        query_lengths.extend([query_length] * pad)
        mask = [1.0] * (RERANK_TOP_M - pad) + [0.0] * pad
    else:
        mask = [1.0] * RERANK_TOP_M

    candidate_doc_ids.append(doc_ids)
    candidate_scores_rows.append(scores)
    candidate_labels_rows.append(labels)
    candidate_mask_rows.append(mask)
    candidate_query_length_rows.append(query_lengths)
    candidate_title_length_rows.append(title_lengths)
    candidate_content_length_rows.append(content_lengths)
    candidate_overlap_count_rows.append(overlap_counts)
    candidate_overlap_ratio_rows.append(overlap_ratios)
    candidate_title_overlap_rows.append(title_overlaps)
    active_qids.append(qid)

candidate_scores = torch.tensor(candidate_scores_rows, dtype=torch.float32)
candidate_labels = torch.tensor(candidate_labels_rows, dtype=torch.float32)
candidate_mask = torch.tensor(candidate_mask_rows, dtype=torch.bool)
candidate_query_length = torch.tensor(candidate_query_length_rows, dtype=torch.float32)
candidate_title_length = torch.tensor(candidate_title_length_rows, dtype=torch.float32)
candidate_content_length = torch.tensor(candidate_content_length_rows, dtype=torch.float32)
candidate_overlap_count = torch.tensor(candidate_overlap_count_rows, dtype=torch.float32)
candidate_overlap_ratio = torch.tensor(candidate_overlap_ratio_rows, dtype=torch.float32)
candidate_title_overlap = torch.tensor(candidate_title_overlap_rows, dtype=torch.float32)

has_relevant_candidate = (candidate_labels > 0).any(dim=1)
active_query_idx = has_relevant_candidate.nonzero(as_tuple=False).squeeze(-1)

rank_qids = [active_qids[i] for i in active_query_idx.tolist()]
rank_doc_ids = [candidate_doc_ids[i] for i in active_query_idx.tolist()]
rank_labels = candidate_labels[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)
rank_scores = candidate_scores[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)
rank_mask = candidate_mask[active_query_idx].to(RERANK_DEVICE)
rank_query_length = candidate_query_length[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)
rank_title_length = candidate_title_length[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)
rank_content_length = candidate_content_length[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)
rank_overlap_count = candidate_overlap_count[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)
rank_overlap_ratio = candidate_overlap_ratio[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)
rank_title_overlap = candidate_title_overlap[active_query_idx].to(RERANK_DEVICE, dtype=torch.float64)

bm25_demo_features, bm25_demo_feature_names = build_bm25_demo_features(rank_scores, rank_mask)
richer_benchmark_features, richer_benchmark_feature_names = build_richer_rank_features(
    rank_scores,
    rank_mask,
    rank_query_length,
    rank_title_length,
    rank_content_length,
    rank_overlap_count,
    rank_overlap_ratio,
    rank_title_overlap,
)

TRACK_FEATURES = {
    "richer_feature_estimator_benchmark": richer_benchmark_features,
    "bm25_shortlist_demo": bm25_demo_features,
}
TRACK_FEATURE_NAMES = {
    "richer_feature_estimator_benchmark": richer_benchmark_feature_names,
    "bm25_shortlist_demo": bm25_demo_feature_names,
}

rank_features = TRACK_FEATURES[EXPERIMENT_TRACK]
rank_feature_names = TRACK_FEATURE_NAMES[EXPERIMENT_TRACK]
EXPERIMENT_TRACK_LABEL = TRACK_TITLE_MAP[EXPERIMENT_TRACK]

print(f"Experiment track: {EXPERIMENT_TRACK_LABEL} ({EXPERIMENT_TRACK})")
print(f"Queries with at least one relevant candidate in top-{RERANK_TOP_M}: {len(rank_qids)} / {len(beir_qids)}")
print("rank_scores shape:", tuple(rank_scores.shape))
print("rank_labels shape:", tuple(rank_labels.shape))
print("selected rank_features shape:", tuple(rank_features.shape))
print("BM25 demo feature dim:", bm25_demo_features.shape[-1])
print("Richer benchmark feature dim:", richer_benchmark_features.shape[-1])
print("Using device:", RERANK_DEVICE)

display(pd.DataFrame({
    "track": ["bm25_shortlist_demo", "richer_feature_estimator_benchmark"],
    "feature_dim": [len(bm25_demo_feature_names), len(richer_benchmark_feature_names)],
}))

display(pd.DataFrame({
    "feature_name": richer_benchmark_feature_names,
    "track": ["richer_feature_estimator_benchmark"] * len(richer_benchmark_feature_names),
}))

sample_idx = 0
print("\nSample active query:")
print(rank_qids[sample_idx], beir_queries_dict[rank_qids[sample_idx]])
print("Top candidate ids:", rank_doc_ids[sample_idx][:10])
print("Top candidate labels:", rank_labels[sample_idx, :10].detach().cpu().tolist())
print("Top candidate BM25 scores:", rank_scores[sample_idx, :10].detach().cpu().tolist())
print("Selected feature names:", rank_feature_names)


In [ ]:
def _discounts(k, device, dtype=torch.float32):
    return 1.0 / torch.log2(torch.arange(2, k + 2, device=device, dtype=dtype))


def dcg_at_k_from_ranked_labels(ranked_labels, k):
    ranked_labels = ranked_labels[..., :k]
    discounts = _discounts(k, ranked_labels.device, ranked_labels.dtype)
    return (ranked_labels * discounts).sum(dim=-1)


def ideal_dcg_at_k(labels, k):
    ideal_ranked = torch.sort(labels, dim=-1, descending=True).values[..., :k]
    discounts = _discounts(k, labels.device, labels.dtype)
    return (ideal_ranked * discounts).sum(dim=-1)


def ndcg_at_k_from_ranked_labels(ranked_labels, full_labels, k):
    dcg = dcg_at_k_from_ranked_labels(ranked_labels, k)
    idcg = ideal_dcg_at_k(full_labels, k)
    while idcg.ndim < dcg.ndim:
        idcg = idcg.unsqueeze(-1)
    zeros = torch.zeros_like(dcg)
    return torch.where(idcg > 0, dcg / idcg, zeros)


def ndcg_rank_weights(labels, k):
    cutoff = min(k, labels.shape[-1])
    discounts = _discounts(cutoff, labels.device, labels.dtype).unsqueeze(0).expand(labels.shape[0], -1)
    idcg = ideal_dcg_at_k(labels, cutoff).unsqueeze(-1)
    safe_idcg = idcg.clamp_min(1e-8)
    zeros = torch.zeros_like(discounts)
    return torch.where(idcg > 0, discounts / safe_idcg, zeros)


def gather_ranked_labels(labels, rankings):
    expanded_labels = labels[:, None, :].expand(-1, rankings.shape[1], -1)
    return torch.gather(expanded_labels, 2, rankings)


def sampled_ndcg(rankings, labels, k):
    ranked_labels = gather_ranked_labels(labels, rankings)
    return ndcg_at_k_from_ranked_labels(ranked_labels, labels, k)


def greedy_rankings_from_scores(scores, mask, topk):
    masked_scores = scores.masked_fill(~mask, float("-inf"))
    return torch.argsort(masked_scores, dim=-1, descending=True)[..., :topk]


def greedy_ndcg(scores, labels, mask, k):
    greedy_rankings = greedy_rankings_from_scores(scores, mask, k)
    ranked_labels = torch.gather(labels, 1, greedy_rankings)
    return ndcg_at_k_from_ranked_labels(ranked_labels, labels, k)


baseline_greedy_ndcg = greedy_ndcg(rank_scores, rank_labels, rank_mask, RERANK_K)
print(f"Mean greedy nDCG@{RERANK_K} from raw BM25 candidate scores: {baseline_greedy_ndcg.mean().item():.4f}")
print("Sample greedy labels:", torch.gather(rank_labels[:1], 1, greedy_rankings_from_scores(rank_scores[:1], rank_mask[:1], RERANK_K)).squeeze(0).detach().cpu().tolist())


In [ ]:
RERANK_MODEL_PARAMS_BY_TRACK = {
    "bm25_shortlist_demo": {
        "hidden_units": [32, 16],
        "activation": "sigmoid",
    },
    "richer_feature_estimator_benchmark": {
        "hidden_units": [64, 32],
        "activation": "relu",
    },
}


def build_activation(name):
    if name == "sigmoid":
        return nn.Sigmoid()
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


class NeuralReranker(nn.Module):
    def __init__(self, feature_dim, model_params=None, dtype=torch.float64):
        super().__init__()
        hidden_units = model_params.get("hidden_units", [])
        activation_name = model_params.get("activation", "relu")

        layers = []
        in_dim = feature_dim
        for hidden_dim in hidden_units:
            layers.append(nn.Linear(in_dim, hidden_dim, dtype=dtype))
            layers.append(build_activation(activation_name))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1, dtype=dtype))

        self.scorer = nn.Sequential(*layers)
        self.output_dtype = dtype
        self.model_params = model_params

    def forward(self, features, mask):
        scores = self.scorer(features.to(self.output_dtype)).squeeze(-1)
        return scores.masked_fill(~mask, float("-inf"))


def make_ranker(seed=RERANK_SEED, feature_dim=None, model_params=None, track_identifier=None):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    track_identifier = EXPERIMENT_TRACK if track_identifier is None else track_identifier
    params = RERANK_MODEL_PARAMS_BY_TRACK[track_identifier] if model_params is None else model_params
    feature_dim = rank_features.shape[-1] if feature_dim is None else feature_dim
    return NeuralReranker(
        feature_dim,
        model_params=params,
    ).to(RERANK_DEVICE)


# Keep the main Plackett-Luce estimators separate from the scalar-reward
# baselines so the thesis-facing comparison stays easy to read.
def dcg_rank_weights(k, device, dtype=torch.float32):
    return _discounts(int(k), device, dtype)


def sampled_dcg(rankings, labels, k):
    ranked_labels = gather_ranked_labels(labels, rankings)
    return dcg_at_k_from_ranked_labels(ranked_labels, k)


def sampled_reward(rankings, labels, k, reward_type="ndcg"):
    if reward_type == "ndcg":
        return sampled_ndcg(rankings, labels, k)
    if reward_type == "dcg":
        return sampled_dcg(rankings, labels, k)
    raise ValueError(f"Unknown reward_type: {reward_type}")


def greedy_dcg(scores, labels, mask, k):
    greedy_rankings = greedy_rankings_from_scores(scores, mask, k)
    ranked_labels = torch.gather(labels, 1, greedy_rankings)
    return dcg_at_k_from_ranked_labels(ranked_labels, k)


def pl_log_probs_for_rankings(logits, rankings, mask):
    batch_size, n_samples, topk = rankings.shape
    expanded_logits = logits[:, None, :].expand(-1, n_samples, -1)
    available = mask[:, None, :].expand(-1, n_samples, -1).clone()
    step_log_probs = []

    for pos in range(topk):
        masked_logits = expanded_logits.masked_fill(~available, float("-inf"))
        step_log_all = F.log_softmax(masked_logits, dim=-1)
        chosen = rankings[:, :, pos]
        chosen_log_prob = torch.gather(step_log_all, 2, chosen.unsqueeze(-1)).squeeze(-1)
        step_log_probs.append(chosen_log_prob)
        available.scatter_(2, chosen.unsqueeze(-1), False)

    return torch.stack(step_log_probs, dim=-1)


def pl_probabilities_for_rankings(logits, rankings, mask):
    batch_size, n_samples, topk = rankings.shape
    expanded_logits = logits[:, None, :].expand(-1, n_samples, -1)
    available = mask[:, None, :].expand(-1, n_samples, -1).clone()
    prob_per_rank = []

    for pos in range(topk):
        masked_logits = expanded_logits.masked_fill(~available, float("-inf"))
        step_probs = F.softmax(masked_logits, dim=-1)
        prob_per_rank.append(step_probs)
        chosen = rankings[:, :, pos]
        available.scatter_(2, chosen.unsqueeze(-1), False)

    return torch.stack(prob_per_rank, dim=2)


def sample_rankings_sequential(logits, mask, n_samples, topk):
    batch_size, n_docs = logits.shape
    expanded_logits = logits[:, None, :].expand(-1, n_samples, -1)
    available = mask[:, None, :].expand(-1, n_samples, -1).clone()
    rankings = []
    step_log_probs = []

    for _ in range(topk):
        masked_logits = expanded_logits.masked_fill(~available, float("-inf"))
        flat_probs = F.softmax(masked_logits, dim=-1).reshape(batch_size * n_samples, n_docs)
        chosen = torch.multinomial(flat_probs, num_samples=1).reshape(batch_size, n_samples)
        chosen_prob = torch.gather(flat_probs, 1, chosen.reshape(-1, 1)).reshape(batch_size, n_samples)
        rankings.append(chosen)
        step_log_probs.append(chosen_prob.clamp_min(1e-12).log())
        available.scatter_(2, chosen.unsqueeze(-1), False)

    return torch.stack(rankings, dim=-1), torch.stack(step_log_probs, dim=-1)


def sample_rankings_pl_gumbel(logits, mask, n_samples, topk):
    batch_size, n_docs = logits.shape
    expanded_logits = logits[:, None, :].expand(-1, n_samples, -1)
    masked_logits = expanded_logits.masked_fill(~mask[:, None, :], float("-inf"))

    # PL-Rank samples rankings with Gumbel perturb-and-sort under the PL model.
    uniform = torch.rand(batch_size, n_samples, n_docs, device=logits.device, dtype=logits.dtype).clamp_(1e-12, 1 - 1e-12)
    gumbel_noise = -torch.log(-torch.log(uniform))
    rankings = torch.argsort(masked_logits + gumbel_noise, dim=-1, descending=True)[..., :topk]
    step_log_probs = pl_log_probs_for_rankings(logits, rankings, mask)
    return rankings, step_log_probs


def sample_rankings_retrieval2_gumbel(logits, mask, n_samples, topk, tau=1.0, hard=False):
    batch_size, n_docs = logits.shape
    expanded_logits = logits[:, None, :].expand(-1, n_samples, -1)
    masked_logits = expanded_logits.masked_fill(~mask[:, None, :], torch.finfo(logits.dtype).min)

    # Scalar-reward PG baselines use gumbel_softmax, argsort, then gather the
    # base log-softmax terms in the sampled order.
    base_log_probs = F.log_softmax(masked_logits, dim=-1)
    sample_probs = F.gumbel_softmax(base_log_probs, tau=tau, hard=hard, dim=-1)
    rankings = torch.argsort(sample_probs, dim=-1, descending=True)[..., :topk]
    gathered_log_probs = torch.gather(base_log_probs, 2, rankings)
    return rankings, gathered_log_probs


def sample_rankings_gumbel_softmax(logits, mask, n_samples, topk, tau=1.0, hard=False):
    return sample_rankings_retrieval2_gumbel(logits, mask, n_samples=n_samples, topk=topk, tau=tau, hard=hard)


def sample_rankings_gumbel_sort(logits, mask, n_samples, topk, tau=1.0):
    if tau != 1.0:
        raise ValueError("`tau` is not part of the PL Gumbel sampler; keep it at 1.0.")
    return sample_rankings_pl_gumbel(logits, mask, n_samples=n_samples, topk=topk)


def sample_rankings(logits, mask, n_samples, topk, sampler="sequential", tau=1.0):
    if sampler == "sequential":
        return sample_rankings_sequential(logits, mask, n_samples=n_samples, topk=topk)
    if sampler in {"pl_gumbel", "gumbel_sort"}:
        return sample_rankings_pl_gumbel(logits, mask, n_samples=n_samples, topk=topk)
    if sampler in {"retrieval2_gumbel", "gumbel"}:
        return sample_rankings_retrieval2_gumbel(logits, mask, n_samples=n_samples, topk=topk, tau=tau)
    raise ValueError(f"Unknown sampler: {sampler}")


def compute_rloo_baseline(rewards):
    if rewards.shape[1] <= 1:
        return torch.zeros_like(rewards)
    return (rewards.sum(dim=1, keepdim=True) - rewards) / (rewards.shape[1] - 1)


def compute_batch_mean_baseline(rewards):
    return torch.full_like(rewards, rewards.mean())


def retrieval2_entropy(log_probs):
    safe_log_probs = torch.clamp(log_probs, min=torch.finfo(log_probs.dtype).min)
    probs = safe_log_probs.exp()
    return -(safe_log_probs * probs).sum(dim=-1)


def sample_rankings_retrieval2_full(logits, mask, n_samples, tau=1.0, hard=False):
    batch_size, n_docs = logits.shape
    masked_logits = logits.masked_fill(~mask, torch.finfo(logits.dtype).min)
    base_log_probs = F.log_softmax(masked_logits, dim=-1)
    entropy = retrieval2_entropy(base_log_probs)

    repeated_log_probs = torch.repeat_interleave(base_log_probs, n_samples, dim=0)
    probs = F.gumbel_softmax(repeated_log_probs, tau=tau, hard=hard, dim=-1)
    full_rankings = torch.argsort(probs, dim=-1, descending=True)
    gathered_log_probs = torch.gather(repeated_log_probs, 1, full_rankings)

    return (
        full_rankings.reshape(batch_size, n_samples, n_docs),
        gathered_log_probs.reshape(batch_size, n_samples, n_docs),
        entropy,
    )


def retrieval2_subranking_rewards(full_rankings, labels, topk, reward_type="ndcg"):
    batch_size, n_samples, n_docs = full_rankings.shape
    flat_rankings = full_rankings.reshape(batch_size * n_samples, n_docs)
    true_rel = torch.repeat_interleave(labels, n_samples, dim=0)
    predicted_rel = torch.gather(true_rel, 1, flat_rankings)
    true_rel = torch.sort(true_rel, dim=1, descending=True)[0]

    top_k_mask = torch.zeros_like(predicted_rel)
    top_k_mask[:, :topk] = 1

    order_ = torch.repeat_interleave(
        torch.arange(1, predicted_rel.shape[1] + 1, device=predicted_rel.device)[None, :],
        predicted_rel.shape[0],
        dim=0,
    )
    map_ = predicted_rel * top_k_mask
    map_ = map_ * order_
    map_ = map_.to(torch.float64)
    map_[map_ > 0] = 1.0 / map_[map_ > 0]
    map_ = map_ / true_rel.sum(dim=1, keepdim=True).clamp_min(1)
    map_ = map_ + torch.sum(map_, dim=-1, keepdim=True) - torch.cumsum(map_, dim=-1)

    dcg_rank = torch.repeat_interleave(
        torch.arange(1, predicted_rel.shape[1] + 1, device=predicted_rel.device)[None, :],
        predicted_rel.shape[0],
        dim=0,
    )

    true_rel = true_rel * top_k_mask
    predicted_rel = predicted_rel * true_rel

    dcg = predicted_rel / torch.log2(dcg_rank + 1)
    idcg = true_rel / torch.log2(dcg_rank + 1)
    dcg = dcg + torch.sum(dcg, dim=-1, keepdim=True) - torch.cumsum(dcg, dim=-1)
    idcg = idcg + torch.sum(idcg, dim=-1, keepdim=True) - torch.cumsum(idcg, dim=-1)

    ndcg = torch.zeros_like(dcg)
    valid = idcg > 0
    ndcg[valid] = dcg[valid] / idcg[valid]

    if reward_type.lower() == "ndcg":
        rewards = ndcg
    elif reward_type.lower() == "map":
        rewards = map_
    else:
        raise NotImplementedError(f"retrieval2 reward_type={reward_type!r} is not implemented")

    return {
        "rewards": rewards.reshape(batch_size, n_samples, n_docs),
        "dcg": dcg.reshape(batch_size, n_samples, n_docs),
        "idcg": idcg.reshape(batch_size, n_samples, n_docs),
        "map": map_.reshape(batch_size, n_samples, n_docs),
        "ndcg": ndcg.reshape(batch_size, n_samples, n_docs),
        "true_rel": true_rel.reshape(batch_size, n_samples, n_docs),
    }


def paper_pg_rank_scalar_reward(full_rankings, labels, topk, reward_type="ndcg"):
    if reward_type.lower() != "ndcg":
        raise NotImplementedError("Paper-faithful PG-Rank currently supports only scalar nDCG reward.")
    batch_size, n_samples, n_docs = full_rankings.shape
    flat_rankings = full_rankings[:, :, :topk].reshape(batch_size * n_samples, topk)
    repeated_labels = torch.repeat_interleave(labels, n_samples, dim=0)
    ranked_labels = torch.gather(repeated_labels, 1, flat_rankings)
    rewards = ndcg_at_k_from_ranked_labels(ranked_labels, topk)
    return rewards.reshape(batch_size, n_samples)


def paper_leave_one_out_baseline(rewards):
    if rewards.shape[1] <= 1:
        return torch.zeros_like(rewards)
    reward_sums = rewards.sum(dim=1, keepdim=True)
    return (reward_sums - rewards) / (rewards.shape[1] - 1)



def build_metric_info(logits, rankings, labels, mask, topk, objective_rewards, objective_name, extras=None):
    sampled_dcg_values = sampled_dcg(rankings, labels, topk)
    sampled_ndcg_values = sampled_ndcg(rankings, labels, topk)
    info = {
        "rankings": rankings,
        "ranking_rewards": objective_rewards,
        "objective_name": objective_name,
        "mean_reward": float(objective_rewards.mean().item()),
        "mean_sampled_dcg": float(sampled_dcg_values.mean().item()),
        "mean_sampled_ndcg": float(sampled_ndcg_values.mean().item()),
        "greedy_dcg": float(greedy_dcg(logits, labels, mask, topk).mean().item()),
        "greedy_ndcg": float(greedy_ndcg(logits, labels, mask, topk).mean().item()),
    }
    if extras:
        info.update(extras)
    return info


def policy_gradient_loss(
    model,
    features,
    labels,
    mask,
    n_samples,
    topk,
    sampler="sequential",
    tau=1.0,
    reward_type="ndcg",
    baseline_type="none",
):
    logits = model(features, mask)
    rankings, step_log_probs = sample_rankings(
        logits,
        mask,
        n_samples=n_samples,
        topk=topk,
        sampler=sampler,
        tau=tau,
    )
    objective_rewards = sampled_reward(rankings, labels, k=topk, reward_type=reward_type)

    if baseline_type == "none":
        baseline = torch.zeros_like(objective_rewards)
    elif baseline_type == "rloo":
        baseline = compute_rloo_baseline(objective_rewards)
    elif baseline_type == "batch_mean":
        baseline = compute_batch_mean_baseline(objective_rewards)
    else:
        raise ValueError(f"Unknown baseline_type: {baseline_type}")

    advantages = objective_rewards - baseline
    loss = -(step_log_probs.sum(dim=-1) * advantages).mean()

    return loss, build_metric_info(
        logits,
        rankings,
        labels,
        mask,
        topk,
        objective_rewards,
        objective_name=reward_type,
        extras={
            "step_log_probs": step_log_probs,
            "baseline": baseline,
            "advantages": advantages,
            "sampler": sampler,
            "reward_type": reward_type,
        },
    )


def reinforce_loss(
    model,
    features,
    labels,
    mask,
    n_samples,
    topk,
    sampler="sequential",
    tau=1.0,
    baseline_type="none",
    reward_type="ndcg",
):
    return policy_gradient_loss(
        model,
        features,
        labels,
        mask,
        n_samples=n_samples,
        topk=topk,
        sampler=sampler,
        tau=tau,
        reward_type=reward_type,
        baseline_type=baseline_type,
    )


def placement_policy_gradient_loss(model, features, labels, mask, n_samples, topk, sampler="pl_gumbel"):
    logits = model(features, mask)
    rankings, step_log_probs = sample_rankings(
        logits,
        mask,
        n_samples=n_samples,
        topk=topk,
        sampler=sampler,
        tau=1.0,
    )
    rank_weights = dcg_rank_weights(topk, labels.device, labels.dtype)
    ranked_labels = gather_ranked_labels(labels, rankings)[..., :topk]
    reward_matrix = ranked_labels * rank_weights.view(1, 1, -1)
    cumulative_rewards = torch.flip(torch.cumsum(torch.flip(reward_matrix, dims=[-1]), dim=-1), dims=[-1])
    loss = -(step_log_probs * cumulative_rewards).sum(dim=-1).mean()
    objective_rewards = reward_matrix.sum(dim=-1)

    return loss, build_metric_info(
        logits,
        rankings,
        labels,
        mask,
        topk,
        objective_rewards,
        objective_name="dcg",
        extras={
            "step_log_probs": step_log_probs,
            "reward_matrix": reward_matrix,
            "cumulative_rewards": cumulative_rewards,
            "sampler": sampler,
        },
    )


def neural_pg_rank_loss(
    model,
    features,
    labels,
    mask,
    n_samples,
    topk,
    tau=0.05,
    reward_type="ndcg",
    baseline_type="paper_loo",
    reward_setup="subranking",
    entropy_coeff=0.01,
    gumbel_softmax_hard=False,
):
    if reward_setup not in {None, "subranking"}:
        raise NotImplementedError("Neural PG-Rank (Eq. 7) uses subranking returns.")
    if baseline_type not in {"paper_loo", "none"}:
        raise NotImplementedError(f"Neural PG-Rank (Eq. 7) baseline_type={baseline_type!r} is not implemented")
    if reward_type.lower() != "ndcg":
        raise NotImplementedError("Neural PG-Rank (Eq. 7) currently supports only nDCG reward.")

    logits = model(features, mask)
    full_rankings, gathered_log_probs, entropy = sample_rankings_retrieval2_full(
        logits,
        mask,
        n_samples=n_samples,
        tau=tau,
        hard=gumbel_softmax_hard,
    )
    reward_info = retrieval2_subranking_rewards(full_rankings, labels, topk=topk, reward_type=reward_type)
    returns = reward_info["rewards"].clone()
    if baseline_type == "paper_loo":
        baseline = compute_rloo_baseline(returns)
    else:
        baseline = torch.zeros_like(returns)
    advantages = returns - baseline

    policy_loss = -(gathered_log_probs * advantages).mean(dim=1).mean()
    entropy_loss = -entropy.mean()
    loss = policy_loss + entropy_coeff * entropy_loss

    sampled_rankings_topk = full_rankings[..., :topk]
    objective_rewards = reward_info[reward_type.lower()][..., 0]
    return loss, build_metric_info(
        logits,
        sampled_rankings_topk,
        labels,
        mask,
        topk,
        objective_rewards,
        objective_name="eq7_ndcg_subranking",
        extras={
            "sampler": "retrieval2_gumbel",
            "tau": tau,
            "baseline_type": baseline_type,
            "reward_type": reward_type,
            "reward_setup": "subranking",
            "entropy_coeff": entropy_coeff,
            "gumbel_softmax_hard": gumbel_softmax_hard,
            "policy_loss": float(policy_loss.detach().cpu()),
            "entropy_loss": float(entropy_loss.detach().cpu()),
            "advantages": advantages,
            "baseline": baseline,
            "returns": returns,
            "returns_mask": reward_info["true_rel"],
            "full_rankings": full_rankings,
            "eq7_style": True,
        },
    )


def pl_rank_1_score_grads(rank_weights, labels, logits, mask, sampled_rankings):
    cutoff = sampled_rankings.shape[-1]
    n_samples = sampled_rankings.shape[1]
    n_docs = logits.shape[-1]

    ranked_labels = gather_ranked_labels(labels, sampled_rankings)[..., :cutoff]
    if rank_weights.ndim == 1:
        rank_weights_exp = rank_weights.view(1, 1, cutoff)
    else:
        rank_weights_exp = rank_weights.unsqueeze(1)
    weighted_labels = ranked_labels * rank_weights_exp
    cumsum_labels = torch.flip(torch.cumsum(torch.flip(weighted_labels, dims=[-1]), dim=-1), dims=[-1])

    positive_terms = torch.zeros(logits.shape[0], n_samples, n_docs, device=logits.device, dtype=logits.dtype)
    positive_terms.scatter_add_(2, sampled_rankings[..., :cutoff], cumsum_labels)
    positive_terms = positive_terms.mean(dim=1)

    prob_per_rank = pl_probabilities_for_rankings(logits, sampled_rankings, mask)
    negative_terms = (prob_per_rank * cumsum_labels.unsqueeze(-1)).sum(dim=2).mean(dim=1)
    return positive_terms - negative_terms


def pl_rank_2_score_grads(rank_weights, labels, logits, mask, sampled_rankings):
    cutoff = sampled_rankings.shape[-1]
    n_samples = sampled_rankings.shape[1]
    n_docs = logits.shape[-1]

    ranked_labels = gather_ranked_labels(labels, sampled_rankings)[..., :cutoff]
    if rank_weights.ndim == 1:
        rank_weights_exp = rank_weights.view(1, 1, cutoff)
        rank_weights_bonus = rank_weights.view(1, 1, cutoff, 1)
    else:
        rank_weights_exp = rank_weights.unsqueeze(1)
        rank_weights_bonus = rank_weights.unsqueeze(1).unsqueeze(-1)
    weighted_labels = ranked_labels * rank_weights_exp
    cumsum_labels = torch.flip(torch.cumsum(torch.flip(weighted_labels, dims=[-1]), dim=-1), dims=[-1])

    positive_terms = torch.zeros(logits.shape[0], n_samples, n_docs, device=logits.device, dtype=logits.dtype)
    if cutoff > 1:
        positive_terms.scatter_add_(2, sampled_rankings[..., :-1], cumsum_labels[..., 1:])
    positive_terms = positive_terms.mean(dim=1)

    prob_per_rank = pl_probabilities_for_rankings(logits, sampled_rankings, mask)
    negative_terms = (prob_per_rank * cumsum_labels.unsqueeze(-1)).sum(dim=2).mean(dim=1)
    relevant_bonus = (prob_per_rank * rank_weights_bonus).sum(dim=2).mean(dim=1) * labels
    return positive_terms - negative_terms + relevant_bonus


def pl_rank_loss(model, features, labels, mask, n_samples, topk, variant="pl_rank_1", sampler="pl_gumbel", reward_type="dcg"):
    logits = model(features, mask)
    rankings, _ = sample_rankings(logits, mask, n_samples=n_samples, topk=topk, sampler=sampler, tau=1.0)
    if reward_type == "dcg":
        rank_weights = dcg_rank_weights(topk, labels.device, labels.dtype)
        objective_rewards = sampled_dcg(rankings, labels, k=topk)
    elif reward_type == "ndcg":
        rank_weights = ndcg_rank_weights(labels, topk)
        objective_rewards = sampled_ndcg(rankings, labels, k=topk)
    else:
        raise ValueError(f"Unknown PL-Rank reward_type: {reward_type}")

    if variant == "pl_rank_1":
        score_grads = pl_rank_1_score_grads(rank_weights, labels, logits, mask, rankings)
    elif variant == "pl_rank_2":
        score_grads = pl_rank_2_score_grads(rank_weights, labels, logits, mask, rankings)
    else:
        raise ValueError(f"Unknown PL-Rank variant: {variant}")

    safe_logits = torch.where(mask, logits, torch.zeros_like(logits))
    loss = -(safe_logits * score_grads.detach()).sum(dim=-1).mean()

    return loss, build_metric_info(
        logits,
        rankings,
        labels,
        mask,
        topk,
        objective_rewards,
        objective_name=reward_type,
        extras={"score_grads": score_grads, "variant": variant, "sampler": sampler, "reward_type": reward_type},
    )


def pl_rank_1_loss(model, features, labels, mask, n_samples, topk, tau=1.0):
    if tau != 1.0:
        raise ValueError("PL-Rank uses the PL Gumbel sampler without a temperature parameter.")
    return pl_rank_loss(model, features, labels, mask, n_samples=n_samples, topk=topk, variant="pl_rank_1")


def pl_rank_2_loss(model, features, labels, mask, n_samples, topk, tau=1.0):
    if tau != 1.0:
        raise ValueError("PL-Rank uses the PL Gumbel sampler without a temperature parameter.")
    return pl_rank_loss(model, features, labels, mask, n_samples=n_samples, topk=topk, variant="pl_rank_2")


def estimator_loss(
    model,
    features,
    labels,
    mask,
    n_samples,
    topk,
    estimator,
    sampler=None,
    tau=1.0,
    baseline_type="none",
    reward_type=None,
    reward_setup=None,
    entropy_coeff=0.0,
    gumbel_softmax_hard=False,
):
    if estimator == "reinforce":
        return policy_gradient_loss(
            model,
            features,
            labels,
            mask,
            n_samples=n_samples,
            topk=topk,
            sampler="sequential" if sampler is None else sampler,
            tau=tau,
            reward_type="ndcg" if reward_type is None else reward_type,
            baseline_type=baseline_type,
        )
    if estimator == "policy_gradient":
        return policy_gradient_loss(
            model,
            features,
            labels,
            mask,
            n_samples=n_samples,
            topk=topk,
            sampler="pl_gumbel" if sampler is None else sampler,
            tau=tau,
            reward_type="ndcg" if reward_type is None else reward_type,
            baseline_type=baseline_type,
        )
    if estimator == "retrieval2_scalar":
        return policy_gradient_loss(
            model,
            features,
            labels,
            mask,
            n_samples=n_samples,
            topk=topk,
            sampler="retrieval2_gumbel" if sampler is None else sampler,
            tau=tau,
            reward_type="ndcg" if reward_type is None else reward_type,
            baseline_type=baseline_type,
        )
    if estimator == "neural_pg_rank":
        return neural_pg_rank_loss(
            model,
            features,
            labels,
            mask,
            n_samples=n_samples,
            topk=topk,
            tau=tau,
            reward_type="ndcg" if reward_type is None else reward_type,
            baseline_type=baseline_type,
            reward_setup=reward_setup,
            entropy_coeff=entropy_coeff,
            gumbel_softmax_hard=gumbel_softmax_hard,
        )
    if estimator == "placement_policy_gradient":
        return placement_policy_gradient_loss(
            model,
            features,
            labels,
            mask,
            n_samples=n_samples,
            topk=topk,
            sampler="pl_gumbel" if sampler is None else sampler,
        )
    if estimator == "pl_rank_1":
        return pl_rank_loss(
            model,
            features,
            labels,
            mask,
            n_samples=n_samples,
            topk=topk,
            variant="pl_rank_1",
            sampler="pl_gumbel" if sampler is None else sampler,
            reward_type="ndcg" if reward_type is None else reward_type,
        )
    if estimator == "pl_rank_2":
        return pl_rank_loss(
            model,
            features,
            labels,
            mask,
            n_samples=n_samples,
            topk=topk,
            variant="pl_rank_2",
            sampler="pl_gumbel" if sampler is None else sampler,
            reward_type="ndcg" if reward_type is None else reward_type,
        )
    raise ValueError(f"Unknown estimator: {estimator}")


ranker = make_ranker()
print(ranker)


## Toy Setwise Evidence Selection

This section adds two toy no-LLM experiments for evidence-set selection. **Option A** keeps a Plackett-Luce policy over passages and uses support-set F1 on the top-`K` prefix. **Option B** treats size-`K` candidate sets as the ranked items and compares an OptiSet-style CE+KL objective against stochastic PL-policy baselines over sets.


In [ ]:

from pathlib import Path

import time

from setwise_toy_utils import (
    build_rank_doc_views,
    build_set_candidate_data,
    load_hotpot_support_title_map,
    mean_support_f1_for_indices,
    sampled_prefix_support_f1,
    support_f1_from_indices,
)

TOY_SET_K = 2
TOY_SET_TOP_D = 10
TOY_SET_DATASET_PATH = Path("dataset/hotpotqa/dev.jsonl")
TOY_SET_TRAIN_FRACTION = 0.8
TOY_SET_SPLIT_SEED = 31415

OPTION_A_N_SAMPLES_GRID = [2, 4, 8, 16]
OPTION_A_RUNS = 8
OPTION_A_VARIANCE_TRIALS = 64
OPTION_A_VARIANCE_MAX_QUERIES = min(256, rank_features.shape[0])
OPTION_A_MINIBATCH_QUERIES = min(32, rank_features.shape[0])
OPTION_A_TARGET_SAMPLE_BUDGET = OPTION_A_MINIBATCH_QUERIES * 150 * min(OPTION_A_N_SAMPLES_GRID)
OPTION_A_CHECKPOINT_COUNT = 20
OPTION_A_LR = 1e-2

OPTION_B_N_SAMPLES_GRID = [2, 4, 8, 16]
OPTION_B_RUNS = 8
OPTION_B_VARIANCE_TRIALS = 64
OPTION_B_VARIANCE_MAX_QUERIES = min(256, rank_features.shape[0])
OPTION_B_MINIBATCH_QUERIES = min(32, rank_features.shape[0])
OPTION_B_TARGET_SAMPLE_BUDGET = OPTION_B_MINIBATCH_QUERIES * 150 * min(OPTION_B_N_SAMPLES_GRID)
OPTION_B_CHECKPOINT_COUNT = 20
OPTION_B_LR = 1e-2
OPTION_B_OPTISET_BETA = 10.0
OPTION_B_OPTISET_KL_WEIGHT = 1.0

TOY_SUPPORT_TITLE_MAP = load_hotpot_support_title_map(TOY_SET_DATASET_PATH)
rank_doc_titles, rank_doc_texts = build_rank_doc_views(beir_qids, flashrag_docs, rank_qids, RERANK_TOP_M)
rank_gold_support_titles = [TOY_SUPPORT_TITLE_MAP.get(qid, set()) for qid in rank_qids]


def maybe_sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def flatten_grads(model):
    flat = []
    for param in model.parameters():
        grad = torch.zeros_like(param) if param.grad is None else param.grad
        flat.append(grad.detach().reshape(-1))
    return torch.cat(flat)


def build_checkpoint_steps(total_steps, checkpoint_count):
    if total_steps <= 1:
        return [0]
    checkpoints = torch.linspace(0, total_steps - 1, steps=min(checkpoint_count, total_steps))
    return sorted({int(round(v.item())) for v in checkpoints})

assert len(rank_doc_titles) == len(rank_qids) == rank_features.shape[0]
assert all(len(titles) == RERANK_TOP_M for titles in rank_doc_titles)


def build_toy_index_split(n_queries, train_fraction=0.8, seed=0):
    generator = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n_queries, generator=generator)
    n_train = max(1, min(n_queries - 1, int(round(n_queries * train_fraction))))
    return perm[:n_train], perm[n_train:]


def subset_option_a(indices):
    idx_list = indices.tolist()
    return {
        'features': rank_features[indices],
        'mask': rank_mask[indices],
        'scores': rank_scores[indices],
        'qids': [rank_qids[i] for i in idx_list],
        'doc_titles': [rank_doc_titles[i] for i in idx_list],
        'gold_titles': [rank_gold_support_titles[i] for i in idx_list],
    }


def subset_option_b(candidate_data, indices):
    idx_list = indices.tolist()
    return {
        'features': candidate_data['features'][indices],
        'utilities': candidate_data['utilities'][indices],
        'mask': candidate_data['mask'][indices],
        'baseline_choice_indices': candidate_data['baseline_choice_indices'][indices],
        'baseline_utilities': candidate_data['baseline_utilities'][indices],
        'oracle_utilities': candidate_data['oracle_utilities'][indices],
        'candidate_sets': candidate_data['candidate_sets'],
        'qids': [rank_qids[i] for i in idx_list],
    }


def option_a_bm25_rankings(split):
    return greedy_rankings_from_scores(split['scores'], split['scores'].ne(float('-inf')) if split['scores'].dtype == torch.float64 else split['mask'], TOY_SET_K)


def support_f1_for_split_rankings(rankings, split, topk=TOY_SET_K):
    prefix_f1 = sampled_prefix_support_f1(rankings[:, None, :], split['doc_titles'], split['gold_titles'], topk)
    return prefix_f1[:, 0, -1]


toy_train_idx, toy_dev_idx = build_toy_index_split(
    len(rank_qids),
    train_fraction=TOY_SET_TRAIN_FRACTION,
    seed=TOY_SET_SPLIT_SEED,
)

option_a_full = subset_option_a(torch.arange(len(rank_qids), device=rank_features.device))
option_a_train = subset_option_a(toy_train_idx)
option_a_dev = subset_option_a(toy_dev_idx)

option_a_bm25_full_rankings = greedy_rankings_from_scores(rank_scores, rank_mask, TOY_SET_K)
option_a_bm25_f1 = sampled_prefix_support_f1(
    option_a_bm25_full_rankings[:, None, :],
    rank_doc_titles,
    rank_gold_support_titles,
    TOY_SET_K,
)[:, 0, -1]

option_b_candidate_data = build_set_candidate_data(
    rank_features,
    rank_scores,
    rank_mask,
    rank_doc_titles,
    rank_doc_texts,
    rank_gold_support_titles,
    max_docs=TOY_SET_TOP_D,
    set_size=TOY_SET_K,
)
option_b_candidate_index = {
    tuple(candidate): idx
    for idx, candidate in enumerate(option_b_candidate_data['candidate_sets'])
}
option_b_candidate_data['baseline_choice_indices'] = torch.tensor(
    [
        option_b_candidate_index[tuple(indices.tolist())]
        for indices in option_b_candidate_data['baseline_set_indices']
    ],
    dtype=torch.long,
    device=rank_features.device,
)
option_b_candidate_data['baseline_utilities'] = option_b_candidate_data['utilities'].gather(
    1,
    option_b_candidate_data['baseline_choice_indices'].unsqueeze(1),
).squeeze(1)

option_b_full = subset_option_b(
    option_b_candidate_data,
    torch.arange(len(rank_qids), device=rank_features.device),
)
option_b_train = subset_option_b(option_b_candidate_data, toy_train_idx)
option_b_dev = subset_option_b(option_b_candidate_data, toy_dev_idx)

print(f'Toy support-title map loaded for {len(TOY_SUPPORT_TITLE_MAP)} HotpotQA queries.')
print(f'Option A train/dev queries: {len(option_a_train["qids"])} / {len(option_a_dev["qids"])}')
print(f'Option B candidate-set feature shape: {tuple(option_b_full["features"].shape)}')
print(f'Option B uses {len(option_b_candidate_data["candidate_sets"])} candidate sets per query (C({TOY_SET_TOP_D}, {TOY_SET_K})).')
print(f'BM25 top-{TOY_SET_K} mean support-set F1: {option_a_bm25_f1.mean().item():.4f}')
print(f'Option B BM25 baseline mean support-set F1: {option_b_candidate_data["baseline_utilities"].mean().item():.4f}')
print(f'Option B oracle mean support-set F1: {option_b_candidate_data["oracle_utilities"].mean().item():.4f}')

assert option_b_full['features'].shape[1] == 45
assert torch.all((option_b_full['utilities'] >= 0.0) & (option_b_full['utilities'] <= 1.0))
assert len(set(option_a_train['qids']) & set(option_a_dev['qids'])) == 0

sample_qid = option_a_dev['qids'][0]
sample_dev_idx = rank_qids.index(sample_qid)
display(pd.DataFrame({
    'sample_qid': [sample_qid],
    'gold_support_titles': [sorted(rank_gold_support_titles[sample_dev_idx])],
    'bm25_top2_titles': [[rank_doc_titles[sample_dev_idx][i] for i in option_a_bm25_full_rankings[sample_dev_idx].detach().cpu().tolist()]],
    'bm25_top2_set_f1': [float(option_a_bm25_f1[sample_dev_idx].item())],
}))


In [ ]:

OPTION_A_METHODS = {
    'Naive REINFORCE': dict(estimator='naive_reinforce', n_samples_grid=OPTION_A_N_SAMPLES_GRID),
    'Neural PG-Rank (Eq. 7)': dict(estimator='pg_rank_eq7', n_samples_grid=OPTION_A_N_SAMPLES_GRID),
}


def option_a_subset(split, batch_idx):
    batch_list = batch_idx.detach().cpu().tolist()
    return {
        'features': split['features'][batch_idx],
        'mask': split['mask'][batch_idx],
        'scores': split['scores'][batch_idx],
        'doc_titles': [split['doc_titles'][i] for i in batch_list],
        'gold_titles': [split['gold_titles'][i] for i in batch_list],
    }


def masked_policy_entropy(logits, mask):
    masked_logits = logits.masked_fill(~mask, float('-inf'))
    probs = F.softmax(masked_logits, dim=-1)
    log_probs = F.log_softmax(masked_logits, dim=-1)
    entropy = -(probs * log_probs).masked_fill(~mask, 0.0).sum(dim=-1)
    return entropy.mean()


def option_a_estimator_loss(model, batch, estimator_name, n_samples):
    logits = model(batch['features'], batch['mask'])
    if estimator_name == 'Naive REINFORCE':
        rankings, step_log_probs = sample_rankings_sequential(logits, batch['mask'], n_samples=n_samples, topk=TOY_SET_K)
        prefix_f1 = sampled_prefix_support_f1(rankings, batch['doc_titles'], batch['gold_titles'], TOY_SET_K)
        rewards = prefix_f1[:, :, -1]
        loss = -(step_log_probs.sum(dim=-1) * rewards.detach()).mean()
        entropy = masked_policy_entropy(logits, batch['mask'])
        return loss, {
            'mean_reward': float(rewards.mean().item()),
            'mean_prefix_f1': float(prefix_f1.mean().item()),
            'mean_final_f1': float(rewards.mean().item()),
            'entropy': float(entropy.item()),
        }

    if estimator_name == 'Neural PG-Rank (Eq. 7)':
        rankings, gathered_log_probs = sample_rankings_retrieval2_gumbel(
            logits,
            batch['mask'],
            n_samples=n_samples,
            topk=TOY_SET_K,
            tau=0.05,
            hard=False,
        )
        prefix_f1 = sampled_prefix_support_f1(rankings, batch['doc_titles'], batch['gold_titles'], TOY_SET_K)
        baseline = compute_rloo_baseline(prefix_f1)
        advantages = prefix_f1 - baseline
        entropy = masked_policy_entropy(logits, batch['mask'])
        policy_loss = -(gathered_log_probs * advantages.detach()).sum(dim=-1).mean()
        loss = policy_loss - 0.01 * entropy
        return loss, {
            'mean_reward': float(prefix_f1[:, :, -1].mean().item()),
            'mean_prefix_f1': float(prefix_f1.mean().item()),
            'mean_final_f1': float(prefix_f1[:, :, -1].mean().item()),
            'entropy': float(entropy.item()),
        }

    raise ValueError(f'Unknown Option A estimator: {estimator_name}')


def option_a_change_fraction(pred_rankings, baseline_rankings):
    pred_rows = pred_rankings.detach().cpu().tolist()
    base_rows = baseline_rankings.detach().cpu().tolist()
    changed = [float(set(pred) != set(base)) for pred, base in zip(pred_rows, base_rows)]
    return float(np.mean(changed))


def evaluate_option_a_split(model, split):
    with torch.no_grad():
        logits = model(split['features'], split['mask'])
        greedy_rankings = greedy_rankings_from_scores(logits, split['mask'], TOY_SET_K)
        greedy_f1 = sampled_prefix_support_f1(
            greedy_rankings[:, None, :],
            split['doc_titles'],
            split['gold_titles'],
            TOY_SET_K,
        )[:, 0, -1]
        baseline_rankings = greedy_rankings_from_scores(split['scores'], split['mask'], TOY_SET_K)
        baseline_f1 = sampled_prefix_support_f1(
            baseline_rankings[:, None, :],
            split['doc_titles'],
            split['gold_titles'],
            TOY_SET_K,
        )[:, 0, -1]
    return {
        'mean_selected_set_f1': float(greedy_f1.mean().item()),
        'mean_bm25_set_f1': float(baseline_f1.mean().item()),
        'delta_over_bm25': float((greedy_f1 - baseline_f1).mean().item()),
        'changed_fraction_vs_bm25': option_a_change_fraction(greedy_rankings, baseline_rankings),
        'average_set_size': float(np.mean([TOY_SET_K] * greedy_f1.shape[0])),
        'perfect_recovery_fraction': float((greedy_f1 >= 0.999).to(torch.float64).mean().item()),
    }


def run_option_a_curve(method_name, n_samples, run_seed, train_split, dev_split):
    model = make_ranker(seed=run_seed, track_identifier=EXPERIMENT_TRACK)
    optimizer = torch.optim.Adam(model.parameters(), lr=OPTION_A_LR)
    n_train = train_split['features'].shape[0]
    minibatch_size = min(OPTION_A_MINIBATCH_QUERIES, n_train)
    total_steps = max(1, int(np.ceil(OPTION_A_TARGET_SAMPLE_BUDGET / (minibatch_size * n_samples))))
    checkpoint_steps = build_checkpoint_steps(total_steps, OPTION_A_CHECKPOINT_COUNT)

    rows = []
    maybe_sync_cuda()
    train_start = time.perf_counter()

    for split_name, split_eval in [('train', evaluate_option_a_split(model, train_split)), ('dev', evaluate_option_a_split(model, dev_split))]:
        rows.append({
            'estimator': method_name,
            'n_samples': n_samples,
            'run': run_seed,
            'step': -1,
            'split': split_name,
            'loss': float('nan'),
            'mean_objective': float('nan'),
            'mean_selected_set_f1': split_eval['mean_selected_set_f1'],
            'delta_over_bm25': split_eval['delta_over_bm25'],
            'changed_fraction_vs_bm25': split_eval['changed_fraction_vs_bm25'],
            'average_set_size': split_eval['average_set_size'],
            'perfect_recovery_fraction': split_eval['perfect_recovery_fraction'],
            'elapsed_time_ms': 0.0,
            'sample_budget_seen': 0,
        })

    cumulative_budget = 0
    for step in range(total_steps):
        optimizer.zero_grad(set_to_none=True)
        step_seed = run_seed + 1000 + step
        torch.manual_seed(step_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(step_seed)

        batch_idx = torch.randperm(n_train, device=train_split['features'].device)[:minibatch_size]
        batch = option_a_subset(train_split, batch_idx)

        maybe_sync_cuda()
        step_start = time.perf_counter()
        loss, info = option_a_estimator_loss(model, batch, method_name, n_samples)
        loss.backward()
        optimizer.step()
        maybe_sync_cuda()

        cumulative_budget += minibatch_size * n_samples
        elapsed_time_ms = (time.perf_counter() - train_start) * 1000.0

        if step in checkpoint_steps:
            for split_name, split_eval in [('train', evaluate_option_a_split(model, train_split)), ('dev', evaluate_option_a_split(model, dev_split))]:
                rows.append({
                    'estimator': method_name,
                    'n_samples': n_samples,
                    'run': run_seed,
                    'step': step,
                    'split': split_name,
                    'loss': float(loss.detach().cpu()),
                    'mean_objective': info['mean_reward'],
                    'mean_selected_set_f1': split_eval['mean_selected_set_f1'],
                    'delta_over_bm25': split_eval['delta_over_bm25'],
                    'changed_fraction_vs_bm25': split_eval['changed_fraction_vs_bm25'],
                    'average_set_size': split_eval['average_set_size'],
                    'perfect_recovery_fraction': split_eval['perfect_recovery_fraction'],
                    'elapsed_time_ms': elapsed_time_ms,
                    'sample_budget_seen': cumulative_budget,
                })

    return pd.DataFrame(rows)


def option_a_variance_experiment(method_name, n_samples_grid, n_trials=64, max_queries=256, query_seed=2468):
    generator = torch.Generator().manual_seed(query_seed)
    idx = torch.randperm(rank_features.shape[0], generator=generator)[:max_queries]
    batch = subset_option_a(idx.to(rank_features.device))
    model = make_ranker(seed=RERANK_SEED, track_identifier=EXPERIMENT_TRACK)
    init_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    rows = []
    for n_samples in n_samples_grid:
        grads = []
        times_ms = []
        rewards = []
        for trial in range(n_trials):
            model.load_state_dict(init_state)
            model.zero_grad(set_to_none=True)
            torch.manual_seed(9000 + trial)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(9000 + trial)
            maybe_sync_cuda()
            start = time.perf_counter()
            loss, info = option_a_estimator_loss(model, batch, method_name, n_samples)
            loss.backward()
            maybe_sync_cuda()
            times_ms.append((time.perf_counter() - start) * 1000.0)
            grads.append(flatten_grads(model).detach().cpu())
            rewards.append(info['mean_reward'])
        grad_var = torch.stack(grads).var(dim=0, unbiased=False)
        rows.append({
            'estimator': method_name,
            'n_samples': n_samples,
            'mean_grad_variance': float(grad_var.mean().item()),
            'max_grad_variance': float(grad_var.max().item()),
            'mean_total_step_time_ms': float(np.mean(times_ms)),
            'mean_reward': float(np.mean(rewards)),
        })
    return pd.DataFrame(rows)


option_a_variance_df = pd.concat(
    [
        option_a_variance_experiment(name, cfg['n_samples_grid'], n_trials=OPTION_A_VARIANCE_TRIALS, max_queries=OPTION_A_VARIANCE_MAX_QUERIES)
        for name, cfg in OPTION_A_METHODS.items()
    ],
    ignore_index=True,
)

option_a_curve_df = pd.concat(
    [
        run_option_a_curve(name, n_samples, RERANK_SEED + 100 * run_idx + n_samples, option_a_train, option_a_dev)
        for name, cfg in OPTION_A_METHODS.items()
        for n_samples in cfg['n_samples_grid']
        for run_idx in range(OPTION_A_RUNS)
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for method_name in OPTION_A_METHODS:
    sub = option_a_variance_df[option_a_variance_df['estimator'] == method_name]
    axes[0, 0].plot(sub['n_samples'], sub['mean_grad_variance'], marker='o', linewidth=2, label=method_name)
    axes[0, 1].plot(sub['mean_total_step_time_ms'], sub['mean_grad_variance'], marker='o', linewidth=2, label=method_name)

low_n = min(OPTION_A_N_SAMPLES_GRID)
low_dev = option_a_curve_df[(option_a_curve_df['split'] == 'dev') & (option_a_curve_df['n_samples'] == low_n)]
low_budget = low_dev.groupby(['estimator', 'sample_budget_seen'], as_index=False).agg(
    mean_selected_set_f1=('mean_selected_set_f1', 'mean'),
    std_selected_set_f1=('mean_selected_set_f1', 'std'),
)
low_time = low_dev.groupby(['estimator', 'step'], as_index=False).agg(
    mean_elapsed_time_ms=('elapsed_time_ms', 'mean'),
    mean_selected_set_f1=('mean_selected_set_f1', 'mean'),
    std_selected_set_f1=('mean_selected_set_f1', 'std'),
)

for method_name in OPTION_A_METHODS:
    sub_budget = low_budget[low_budget['estimator'] == method_name]
    axes[1, 0].plot(sub_budget['sample_budget_seen'], sub_budget['mean_selected_set_f1'], marker='o', linewidth=2, label=method_name)
    if len(sub_budget) > 1:
        axes[1, 0].fill_between(
            sub_budget['sample_budget_seen'],
            sub_budget['mean_selected_set_f1'] - sub_budget['std_selected_set_f1'].fillna(0.0),
            sub_budget['mean_selected_set_f1'] + sub_budget['std_selected_set_f1'].fillna(0.0),
            alpha=0.15,
        )
    sub_time = low_time[low_time['estimator'] == method_name]
    axes[1, 1].plot(sub_time['mean_elapsed_time_ms'], sub_time['mean_selected_set_f1'], marker='o', linewidth=2, label=method_name)
    if len(sub_time) > 1:
        axes[1, 1].fill_between(
            sub_time['mean_elapsed_time_ms'],
            sub_time['mean_selected_set_f1'] - sub_time['std_selected_set_f1'].fillna(0.0),
            sub_time['mean_selected_set_f1'] + sub_time['std_selected_set_f1'].fillna(0.0),
            alpha=0.15,
        )

for ax in [axes[1, 0], axes[1, 1]]:
    ax.axhline(float(option_a_bm25_f1.mean().item()), linestyle='--', color='black', linewidth=1.5, label='BM25 top-k baseline')

axes[0, 0].set_xscale('log', base=2)
axes[0, 0].set_yscale('log')
axes[0, 0].set_title('Option A Variance')
axes[0, 0].set_xlabel('MC samples per query')
axes[0, 0].set_ylabel('Mean gradient variance')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Option A Compute vs Variance')
axes[0, 1].set_xlabel('Mean step time (ms)')
axes[0, 1].set_ylabel('Mean gradient variance')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].set_title(f'Option A Dev Set F1 vs Budget (N={low_n})')
axes[1, 0].set_xlabel('Sample budget seen')
axes[1, 0].set_ylabel('Mean selected-set F1')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].set_title(f'Option A Dev Set F1 vs Time (N={low_n})')
axes[1, 1].set_xlabel('Elapsed training time (ms)')
axes[1, 1].set_ylabel('Mean selected-set F1')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

option_a_final_dev = option_a_curve_df[option_a_curve_df['split'] == 'dev'].sort_values(['estimator', 'n_samples', 'run', 'sample_budget_seen']).groupby(['estimator', 'n_samples', 'run']).tail(1)
option_a_summary = option_a_final_dev.groupby(['estimator', 'n_samples'], as_index=False).agg(
    final_dev_set_f1=('mean_selected_set_f1', 'mean'),
    final_dev_delta_over_bm25=('delta_over_bm25', 'mean'),
    perfect_recovery_fraction=('perfect_recovery_fraction', 'mean'),
    changed_fraction_vs_bm25=('changed_fraction_vs_bm25', 'mean'),
)

display(option_a_summary)


In [ ]:

OPTION_B_METHODS = {
    'OptiSet CE+KL': dict(kind='optiset', n_samples_grid=[1]),
    'REINFORCE-over-sets': dict(kind='stochastic', estimator='policy_gradient', sampler='sequential', reward_type='dcg', n_samples_grid=OPTION_B_N_SAMPLES_GRID),
    'PL-over-sets (PL-Rank-1)': dict(kind='stochastic', estimator='pl_rank_1', sampler='pl_gumbel', reward_type='dcg', n_samples_grid=OPTION_B_N_SAMPLES_GRID),
}


def make_option_b_ranker(seed):
    return make_ranker(
        seed=seed,
        feature_dim=option_b_full['features'].shape[-1],
        track_identifier=EXPERIMENT_TRACK,
    )


def option_b_entropy(logits, mask):
    masked_logits = logits.masked_fill(~mask, float('-inf'))
    probs = F.softmax(masked_logits, dim=-1)
    log_probs = F.log_softmax(masked_logits, dim=-1)
    entropy = -(probs * log_probs).masked_fill(~mask, 0.0).sum(dim=-1)
    return entropy.mean()


def option_b_optiset_loss(model, features, utilities, mask, beta=OPTION_B_OPTISET_BETA, kl_weight=OPTION_B_OPTISET_KL_WEIGHT):
    logits = model(features, mask)
    masked_logits = logits.masked_fill(~mask, float('-inf'))
    target_scores = torch.where(mask, utilities, torch.full_like(utilities, float('-inf')))
    best_idx = target_scores.argmax(dim=1)

    log_probs = F.log_softmax(masked_logits, dim=-1)
    ce = F.nll_loss(log_probs, best_idx)

    target_logits = torch.where(mask, beta * utilities, torch.full_like(utilities, float('-inf')))
    target_probs = F.softmax(target_logits, dim=-1)
    target_log_probs = torch.log(target_probs.clamp_min(1e-12))
    kl = (target_probs * (target_log_probs - log_probs)).masked_fill(~mask, 0.0).sum(dim=-1).mean()
    loss = ce + kl_weight * kl

    chosen = masked_logits.argmax(dim=1)
    selected_utilities = utilities.gather(1, chosen.unsqueeze(1)).squeeze(1)
    return loss, {
        'mean_reward': float(selected_utilities.mean().item()),
        'ce': float(ce.detach().cpu()),
        'kl': float(kl.detach().cpu()),
        'mean_entropy': float(option_b_entropy(logits, mask).detach().cpu()),
    }


def option_b_stochastic_loss(model, features, utilities, mask, cfg, n_samples):
    loss, info = estimator_loss(
        model,
        features,
        utilities,
        mask,
        n_samples=n_samples,
        topk=1,
        estimator=cfg['estimator'],
        sampler=cfg['sampler'],
        reward_type=cfg['reward_type'],
        baseline_type='none',
        tau=1.0,
    )
    info['mean_reward'] = info['mean_reward']
    return loss, info


def evaluate_option_b_split(model, split):
    with torch.no_grad():
        logits = model(split['features'], split['mask'])
        masked_logits = logits.masked_fill(~split['mask'], float('-inf'))
        chosen = masked_logits.argmax(dim=1)
        selected_utilities = split['utilities'].gather(1, chosen.unsqueeze(1)).squeeze(1)
        baseline_utilities = split['baseline_utilities']
        oracle_utilities = split['oracle_utilities']
    return {
        'mean_selected_set_f1': float(selected_utilities.mean().item()),
        'mean_bm25_set_f1': float(baseline_utilities.mean().item()),
        'delta_over_bm25': float((selected_utilities - baseline_utilities).mean().item()),
        'oracle_gap': float((oracle_utilities - selected_utilities).mean().item()),
        'changed_fraction_vs_bm25': float((chosen != split['baseline_choice_indices']).to(torch.float64).mean().item()),
        'average_set_size': float(np.mean([TOY_SET_K] * selected_utilities.shape[0])),
        'perfect_recovery_fraction': float((selected_utilities >= 0.999).to(torch.float64).mean().item()),
    }


def option_b_subset(split, batch_idx):
    return {
        'features': split['features'][batch_idx],
        'utilities': split['utilities'][batch_idx],
        'mask': split['mask'][batch_idx],
        'baseline_choice_indices': split['baseline_choice_indices'][batch_idx],
        'baseline_utilities': split['baseline_utilities'][batch_idx],
        'oracle_utilities': split['oracle_utilities'][batch_idx],
    }


def run_option_b_curve(method_name, cfg, n_samples, run_seed, train_split, dev_split):
    model = make_option_b_ranker(run_seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=OPTION_B_LR)
    n_train = train_split['features'].shape[0]
    minibatch_size = min(OPTION_B_MINIBATCH_QUERIES, n_train)
    effective_samples = max(1, n_samples)
    total_steps = max(1, int(np.ceil(OPTION_B_TARGET_SAMPLE_BUDGET / (minibatch_size * effective_samples))))
    checkpoint_steps = build_checkpoint_steps(total_steps, OPTION_B_CHECKPOINT_COUNT)

    rows = []
    maybe_sync_cuda()
    train_start = time.perf_counter()

    for split_name, split_eval in [('train', evaluate_option_b_split(model, train_split)), ('dev', evaluate_option_b_split(model, dev_split))]:
        rows.append({
            'estimator': method_name,
            'n_samples': n_samples,
            'run': run_seed,
            'step': -1,
            'split': split_name,
            'loss': float('nan'),
            'mean_objective': float('nan'),
            'mean_selected_set_f1': split_eval['mean_selected_set_f1'],
            'delta_over_bm25': split_eval['delta_over_bm25'],
            'oracle_gap': split_eval['oracle_gap'],
            'changed_fraction_vs_bm25': split_eval['changed_fraction_vs_bm25'],
            'average_set_size': split_eval['average_set_size'],
            'perfect_recovery_fraction': split_eval['perfect_recovery_fraction'],
            'elapsed_time_ms': 0.0,
            'sample_budget_seen': 0,
        })

    cumulative_budget = 0
    for step in range(total_steps):
        optimizer.zero_grad(set_to_none=True)
        step_seed = run_seed + 4000 + step
        torch.manual_seed(step_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(step_seed)

        batch_idx = torch.randperm(n_train, device=train_split['features'].device)[:minibatch_size]
        batch = option_b_subset(train_split, batch_idx)

        maybe_sync_cuda()
        if cfg['kind'] == 'optiset':
            loss, info = option_b_optiset_loss(model, batch['features'], batch['utilities'], batch['mask'])
            effective_step_samples = 1
        else:
            loss, info = option_b_stochastic_loss(model, batch['features'], batch['utilities'], batch['mask'], cfg, n_samples)
            effective_step_samples = n_samples
        loss.backward()
        optimizer.step()
        maybe_sync_cuda()

        cumulative_budget += minibatch_size * effective_step_samples
        elapsed_time_ms = (time.perf_counter() - train_start) * 1000.0

        if step in checkpoint_steps:
            for split_name, split_eval in [('train', evaluate_option_b_split(model, train_split)), ('dev', evaluate_option_b_split(model, dev_split))]:
                rows.append({
                    'estimator': method_name,
                    'n_samples': n_samples,
                    'run': run_seed,
                    'step': step,
                    'split': split_name,
                    'loss': float(loss.detach().cpu()),
                    'mean_objective': info['mean_reward'],
                    'mean_selected_set_f1': split_eval['mean_selected_set_f1'],
                    'delta_over_bm25': split_eval['delta_over_bm25'],
                    'oracle_gap': split_eval['oracle_gap'],
                    'changed_fraction_vs_bm25': split_eval['changed_fraction_vs_bm25'],
                    'average_set_size': split_eval['average_set_size'],
                    'perfect_recovery_fraction': split_eval['perfect_recovery_fraction'],
                    'elapsed_time_ms': elapsed_time_ms,
                    'sample_budget_seen': cumulative_budget,
                })

    return pd.DataFrame(rows)


def option_b_variance_experiment(method_name, cfg, n_samples_grid, n_trials=64, max_queries=256, query_seed=97531):
    generator = torch.Generator().manual_seed(query_seed)
    idx = torch.randperm(option_b_full['features'].shape[0], generator=generator)[:max_queries]
    batch = option_b_subset(option_b_full, idx.to(option_b_full['features'].device))
    model = make_option_b_ranker(RERANK_SEED)
    init_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    rows = []
    for n_samples in n_samples_grid:
        grads = []
        times_ms = []
        rewards = []
        for trial in range(n_trials):
            model.load_state_dict(init_state)
            model.zero_grad(set_to_none=True)
            torch.manual_seed(12000 + trial)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(12000 + trial)
            maybe_sync_cuda()
            start = time.perf_counter()
            if cfg['kind'] == 'optiset':
                loss, info = option_b_optiset_loss(model, batch['features'], batch['utilities'], batch['mask'])
            else:
                loss, info = option_b_stochastic_loss(model, batch['features'], batch['utilities'], batch['mask'], cfg, n_samples)
            loss.backward()
            maybe_sync_cuda()
            grads.append(flatten_grads(model).detach().cpu())
            times_ms.append((time.perf_counter() - start) * 1000.0)
            rewards.append(info['mean_reward'])
        grad_var = torch.stack(grads).var(dim=0, unbiased=False)
        rows.append({
            'estimator': method_name,
            'n_samples': n_samples,
            'mean_grad_variance': float(grad_var.mean().item()),
            'mean_total_step_time_ms': float(np.mean(times_ms)),
            'mean_reward': float(np.mean(rewards)),
        })
    return pd.DataFrame(rows)


option_b_variance_df = pd.concat(
    [
        option_b_variance_experiment(
            method_name,
            cfg,
            cfg['n_samples_grid'],
            n_trials=OPTION_B_VARIANCE_TRIALS,
            max_queries=OPTION_B_VARIANCE_MAX_QUERIES,
        )
        for method_name, cfg in OPTION_B_METHODS.items()
    ],
    ignore_index=True,
)

option_b_curve_df = pd.concat(
    [
        run_option_b_curve(method_name, cfg, n_samples, RERANK_SEED + 500 * run_idx + 10 * n_samples, option_b_train, option_b_dev)
        for method_name, cfg in OPTION_B_METHODS.items()
        for n_samples in cfg['n_samples_grid']
        for run_idx in range(OPTION_B_RUNS)
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for method_name, cfg in OPTION_B_METHODS.items():
    sub = option_b_variance_df[option_b_variance_df['estimator'] == method_name]
    axes[0, 0].plot(sub['n_samples'], sub['mean_grad_variance'], marker='o', linewidth=2, label=method_name)
    axes[0, 1].plot(sub['mean_total_step_time_ms'], sub['mean_grad_variance'], marker='o', linewidth=2, label=method_name)

low_n = min(OPTION_B_N_SAMPLES_GRID)
low_dev = option_b_curve_df[
    (option_b_curve_df['split'] == 'dev')
    & (
        ((option_b_curve_df['estimator'] == 'OptiSet CE+KL') & (option_b_curve_df['n_samples'] == 1))
        | ((option_b_curve_df['estimator'] != 'OptiSet CE+KL') & (option_b_curve_df['n_samples'] == low_n))
    )
]
low_budget = low_dev.groupby(['estimator', 'sample_budget_seen'], as_index=False).agg(
    mean_selected_set_f1=('mean_selected_set_f1', 'mean'),
    std_selected_set_f1=('mean_selected_set_f1', 'std'),
)
low_time = low_dev.groupby(['estimator', 'step'], as_index=False).agg(
    mean_elapsed_time_ms=('elapsed_time_ms', 'mean'),
    mean_selected_set_f1=('mean_selected_set_f1', 'mean'),
    std_selected_set_f1=('mean_selected_set_f1', 'std'),
)

for method_name in OPTION_B_METHODS:
    sub_budget = low_budget[low_budget['estimator'] == method_name]
    axes[1, 0].plot(sub_budget['sample_budget_seen'], sub_budget['mean_selected_set_f1'], marker='o', linewidth=2, label=method_name)
    if len(sub_budget) > 1:
        axes[1, 0].fill_between(
            sub_budget['sample_budget_seen'],
            sub_budget['mean_selected_set_f1'] - sub_budget['std_selected_set_f1'].fillna(0.0),
            sub_budget['mean_selected_set_f1'] + sub_budget['std_selected_set_f1'].fillna(0.0),
            alpha=0.15,
        )
    sub_time = low_time[low_time['estimator'] == method_name]
    axes[1, 1].plot(sub_time['mean_elapsed_time_ms'], sub_time['mean_selected_set_f1'], marker='o', linewidth=2, label=method_name)
    if len(sub_time) > 1:
        axes[1, 1].fill_between(
            sub_time['mean_elapsed_time_ms'],
            sub_time['mean_selected_set_f1'] - sub_time['std_selected_set_f1'].fillna(0.0),
            sub_time['mean_selected_set_f1'] + sub_time['std_selected_set_f1'].fillna(0.0),
            alpha=0.15,
        )

for ax in [axes[1, 0], axes[1, 1]]:
    ax.axhline(float(option_b_candidate_data['baseline_utilities'].mean().item()), linestyle='--', color='black', linewidth=1.5, label='BM25 top-k baseline')
    ax.axhline(float(option_b_candidate_data['oracle_utilities'].mean().item()), linestyle=':', color='gray', linewidth=1.5, label='Oracle best set')

axes[0, 0].set_xscale('log', base=2)
axes[0, 0].set_yscale('log')
axes[0, 0].set_title('Option B Variance')
axes[0, 0].set_xlabel('MC samples per query')
axes[0, 0].set_ylabel('Mean gradient variance')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Option B Compute vs Variance')
axes[0, 1].set_xlabel('Mean step time (ms)')
axes[0, 1].set_ylabel('Mean gradient variance')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].set_title('Option B Dev Set F1 vs Budget')
axes[1, 0].set_xlabel('Optimization budget seen')
axes[1, 0].set_ylabel('Mean selected-set F1')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].set_title('Option B Dev Set F1 vs Time')
axes[1, 1].set_xlabel('Elapsed training time (ms)')
axes[1, 1].set_ylabel('Mean selected-set F1')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

option_b_final_dev = option_b_curve_df[option_b_curve_df['split'] == 'dev'].sort_values(['estimator', 'n_samples', 'run', 'sample_budget_seen']).groupby(['estimator', 'n_samples', 'run']).tail(1)
option_b_summary = option_b_final_dev.groupby(['estimator', 'n_samples'], as_index=False).agg(
    final_dev_set_f1=('mean_selected_set_f1', 'mean'),
    final_dev_delta_over_bm25=('delta_over_bm25', 'mean'),
    oracle_gap=('oracle_gap', 'mean'),
    perfect_recovery_fraction=('perfect_recovery_fraction', 'mean'),
    changed_fraction_vs_bm25=('changed_fraction_vs_bm25', 'mean'),
)

display(option_b_summary)
